# Google Play Store: App Market Analysis
### Report Notebook — SQL Findings, Visualizations & Recommendations

**Business case:** We're an app development agency deciding on pricing strategy and category for our next app launch. This notebook connects to our cleaned `GoogleApps` MySQL database, runs our five analysis queries, visualizes the results, and interprets what they mean for the launch decision.

**Note:** this notebook is intentionally separate from `data_cleaning.ipynb`. All cleaning, type-fixing, and CSV export logic lives there — this notebook only reads from the already-loaded MySQL database.

**Guiding questions:**
1. Should our next app be free or paid?
2. Which category offers the best mix of high ratings and high install volume?
3. Does review volume tell us anything about app quality?


## Setup
Connect to the `GoogleApps` MySQL database and load our plotting libraries.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# --- Update these to match your local MySQL setup ---
DB_USER = "root"
DB_PASSWORD = "your_password_here"
DB_HOST = "localhost"
DB_NAME = "GoogleApps"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print("Connected to", DB_NAME)

## Q1 — Which categories have the highest average rating?

**Business question:** Which categories should we prioritize for consistently higher ratings?

In [ ]:
query_q1 = """
SELECT
    c.category_name AS category,
    AVG(a.rating)    AS avg_rating,
    STDDEV(a.rating) AS rating_stddev,
    COUNT(a.app_id)  AS num_apps
FROM apps AS a
INNER JOIN categories AS c ON a.category_id = c.category_id
WHERE a.rating IS NOT NULL
GROUP BY category
ORDER BY avg_rating DESC;
"""

df_q1 = pd.read_sql(query_q1, engine)
df_q1.head(10)

In [ ]:
top10 = df_q1.head(10).sort_values("avg_rating")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top10["category"], top10["avg_rating"], xerr=top10["rating_stddev"],
        color="#3B82F6", capsize=3, ecolor="#475569")
ax.set_xlabel("Average rating (error bars = rating std. dev.)")
ax.set_title("Top 10 categories by average rating", loc="left", fontweight="bold")
ax.set_xlim(0, 5)
plt.tight_layout()
plt.show()

**Finding:** Ratings cluster tightly across all 33 categories — the full range is roughly 4.01 to 4.39, with Education narrowly on top. Notice the error bars: within-category variation (std. dev.) is often *larger* than the gap between categories.

**So what:** category choice is a weak lever for rating. Execution — app quality, onboarding, support — matters far more than which category we launch in.

## Q2 — Do paid apps have higher sentiment than free apps?

**Business question:** Do paid apps generate more positive user sentiment than free apps, measured via `sentiment_polarity` (-1 = most negative, +1 = most positive)?

In [ ]:
query_q2 = """
SELECT
    a.type,
    AVG(r.sentiment_polarity) AS avg_sentiment_polarity,
    COUNT(r.review_id)        AS num_reviews
FROM apps AS a
INNER JOIN reviews AS r ON a.app_id = r.app_id
GROUP BY a.type;
"""

df_q2 = pd.read_sql(query_q2, engine)
df_q2

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
colors = ["#3B82F6", "#8B5CF6"]
bars = ax.bar(df_q2["type"], df_q2["avg_sentiment_polarity"], color=colors, width=0.5)

for bar, val in zip(bars, df_q2["avg_sentiment_polarity"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.3f}",
            ha="center", fontweight="bold")

ax.set_ylabel("Average sentiment polarity")
ax.set_title("Sentiment polarity: Free vs. Paid", loc="left", fontweight="bold")
ax.set_ylim(0, max(df_q2["avg_sentiment_polarity"]) * 1.4)
plt.tight_layout()
plt.show()

**Finding:** Paid apps score marginally higher in sentiment, but the gap is narrow — both types remain solidly in positive territory.

**So what:** sentiment alone is not a decisive factor for the free-vs-paid pricing decision.

## Q3 — What's the average price per category (paid apps)?

**Business question:** Which app categories can realistically sustain a premium paid price?

In [ ]:
query_q3 = """
SELECT
    c.category_name AS category_name,
    AVG(a.price)     AS avg_price,
    MIN(a.price)     AS min_price,
    MAX(a.price)     AS max_price,
    COUNT(a.app_id)  AS num_paid_apps
FROM apps AS a
INNER JOIN categories AS c ON a.category_id = c.category_id
WHERE a.type = 'Paid'
GROUP BY category_name
ORDER BY avg_price DESC;
"""

df_q3 = pd.read_sql(query_q3, engine)

# Flag categories whose average is likely unreliable: a single paid app,
# or a max price so high ($300+) it's almost certainly an outlier/novelty app.
df_q3["unreliable"] = (df_q3["min_price"] == df_q3["max_price"]) | (df_q3["max_price"] >= 300)
df_q3.head(10)

In [ ]:
plot_df = df_q3.sort_values("avg_price", ascending=True)
colors = ["#B4B2A9" if u else "#378ADD" for u in plot_df["unreliable"]]

fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(plot_df["category_name"], plot_df["avg_price"], color=colors)

# .clip(lower=0) guards against tiny floating-point rounding artifacts
# (e.g. a category with a single paid app can compute avg_price as
# a hair below min_price due to floating-point division, which
# matplotlib's errorbar() rejects as a negative error value)
lower_err = (plot_df["avg_price"] - plot_df["min_price"]).clip(lower=0)
upper_err = (plot_df["max_price"] - plot_df["avg_price"]).clip(lower=0)
ax.errorbar(plot_df["avg_price"], plot_df["category_name"],
            xerr=[lower_err, upper_err], fmt="none", ecolor="#5F5E5A", capsize=2)

ax.set_xlabel("Price (USD) — bar = average, whiskers = min-max range")
ax.set_title("Average paid-app price by category\n(grey = unreliable: 1 paid app, or $300+ outlier)",
              loc="left", fontweight="bold")
plt.tight_layout()
plt.show()

**Finding:** Reliable pricing sits around **$9–$15** (Business, Medical, Productivity). Headline highs — Finance, Lifestyle, Events — are misleading: some have just 1-2 paid apps, or are pulled up by rare $300+ outlier apps.

**So what:** set pricing benchmarks off the stable mid-tier categories, not the highest raw average.

## Q4 — Which categories have the highest total installs?

**Business question:** Which categories give the agency the most reach?

In [ ]:
query_q4 = """
SELECT
    c.category_name      AS category_name,
    SUM(a.installs)       AS tot_install_count
FROM apps AS a
INNER JOIN categories AS c ON a.category_id = c.category_id
GROUP BY category_name
ORDER BY tot_install_count DESC;
"""

df_q4 = pd.read_sql(query_q4, engine)
df_q4.head(10)

In [ ]:
plot_df = df_q4.sort_values("tot_install_count", ascending=True)
highlight = {"GAME": "#3B82F6", "COMMUNICATION": "#3B82F6", "FINANCE": "#D85A30"}
colors = [highlight.get(cat, "#B4B2A9") for cat in plot_df["category_name"]]

fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(plot_df["category_name"], plot_df["tot_install_count"], color=colors)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: f"{x/1e9:.1f}B" if x >= 1e9 else f"{x/1e6:.0f}M"))
ax.set_xlabel("Total installs")
ax.set_title("Total installs by category\n(blue = reach leaders, orange = highest-priced category)",
              loc="left", fontweight="bold")
plt.tight_layout()
plt.show()

**Finding:** Game and Communication dominate reach by a wide margin. Finance — the highest-priced category from Q3 — ranks only around 15th in installs.

**So what:** high price does not mean high reach. Categories that scale to huge install bases tend to rely on free access, not premium pricing.

## Q5 — Does review volume correlate with rating?

**Business question:** Does review volume signal a higher-quality app?

In [ ]:
query_q5 = """
SELECT
    CASE
        WHEN a.reviews_count >= (SELECT AVG(reviews_count) FROM apps)
            THEN 'Above average review count'
        ELSE 'Below average review count'
    END AS review_volume_bucket,
    COUNT(a.app_id) AS num_apps,
    AVG(a.rating)   AS avg_rating
FROM apps AS a
WHERE a.rating IS NOT NULL
GROUP BY review_volume_bucket;
"""

df_q5 = pd.read_sql(query_q5, engine)
df_q5

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = ["#B4B2A9", "#3B82F6"]
bars = ax.bar(df_q5["review_volume_bucket"], df_q5["avg_rating"], color=colors, width=0.5)

for bar, val, n in zip(bars, df_q5["avg_rating"], df_q5["num_apps"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.05, f"{val:.2f}",
            ha="center", fontweight="bold", fontsize=13)
    ax.text(bar.get_x() + bar.get_width() / 2, 0.15, f"n = {n:,}",
            ha="center", color="white", fontsize=9)

ax.set_ylim(0, 5)
ax.set_ylabel("Average rating")
ax.set_title("Review volume vs. average rating", loc="left", fontweight="bold")
plt.xticks(rotation=5)
plt.tight_layout()
plt.show()

**Finding:** Apps with above-average review counts rate modestly higher — a real but small gap, and only around 10% of apps clear that "above average" bar (the distribution is heavily skewed by a handful of blockbuster apps).

**So what:** treat review volume as a weak, secondary signal — not proof that more reviews *cause* better ratings.

## Conclusions & Recommendations

Bringing all five findings together:

1. **Adopt a freemium pricing model.** Sentiment barely differs between free and paid apps (Q2), but the categories with the biggest reach are all low-price or free (Q3 vs. Q4) — scale requires free access.
2. **Separate the rating decision from the reach decision.** Education leads on rating, but the spread across categories is small (Q1). Game and Communication dominate installs (Q4). Don't expect one category to win on both fronts — pick based on product fit, then benchmark against the right goal.
3. **Treat reviews as engagement, not proof of quality.** More reviews associate with modestly higher ratings (Q5), but the relationship is weak and not causal. Improve ratings directly through product quality, onboarding, and support — don't rely on review volume alone.

**Limitations:** these are correlational findings from a single historical snapshot of the Play Store, not a controlled experiment — none of the relationships above should be read as causal. Category-level averages can also be skewed by small sample sizes or outlier apps, which is why several of the visualizations above explicitly flag or account for that.
